# M2 — Data Understanding & Audit Kualitas Citra
**Proyek:** Klasifikasi Penyakit Daun Padi (*Rice Leaf Disease Classification*)  
**Dataset:** `penyakit padi/` (Sethy et al., 2020 — Mendeley Data, 4 Kelas: `Bacterialblight`, `Blast`, `Brownspot`, `Tungro`)

Notebook ini melakukan audit menyeluruh terhadap 5.932 citra pada folder `penyakit padi/`:
1. Memeriksa jumlah file per kelas dan memastikan tidak ada file rusak (*corrupt*).
2. Memeriksa distribusi resolusi piksel dan mode warna (`RGB` vs `RGBA`).
3. Mendeteksi **duplikat file identik (MD5 Hash)** dan **grup spesimen daun fisik (`leaf_group`)** untuk mencegah *data leakage*.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src.data_loader import scan_dataset, get_dataset_summary
from src import DATA_PROCESSED_DIR

df_meta = pd.read_csv(DATA_PROCESSED_DIR / "metadata_audit.csv")
audit = get_dataset_summary(df_meta)
print("Total Citra Mentah :", audit["total_files"])
print("Citra Corrupt      :", audit["corrupt_count"])
print("Duplikat MD5       :", audit["duplicate_count"], f"({audit['duplicate_pct']}%)")
print("Citra Unik         :", audit["unique_count"])
print("Spesimen Daun Unik :", audit["unique_leaf_specimens"])
df_meta.head(10)

Total Citra Mentah : 5932
Citra Corrupt      : 0
Duplikat MD5       : 1138 (19.18%)
Citra Unik         : 4794
Spesimen Daun Unik : 858


In [2]:
# Perbandingan distribusi kelas sebelum vs sesudah hapus duplikat MD5
dist_df = pd.DataFrame({
    "Total Mentah": pd.Series(audit["per_class_raw"]),
    "Duplikat MD5": pd.Series(audit["per_class_dup"]).fillna(0).astype(int),
    "Citra Unik": pd.Series(audit["per_class_unique"]),
    "Spesimen Daun Fisik (leaf_group)": pd.Series(audit["per_class_leaf_specimens"]),
})
dist_df["% Duplikat"] = (dist_df["Duplikat MD5"] / dist_df["Total Mentah"] * 100).round(2)
print(dist_df)

Total Mentah  Duplikat MD5  Citra Unik  Spesimen Daun Fisik (leaf_group)  % Duplikat
Bacterialblight          1584           258        1326                               221       16.29
Blast                    1440           480         960                               160       33.33
Brownspot                1600           400        1200                               200       25.00
Tungro                   1308             0        1308                               277        0.00


### Kesimpulan Audit Data Understanding (M2)
1. **Tidak ada file rusak (0 corrupt)** dari 5.932 citra.
2. **Mode Warna:** Terdapat **156 citra berformat `RGBA` (4 kanal)** dan **5.776 citra `RGB` (3 kanal)**. Pada tahap preprocessing, citra `RGBA` wajib dikonversi menjadi `RGB` 3 kanal di atas latar putih agar dimensi kanal seragam.
3. **Resolusi Citra:** Sebanyak 4.624 citra (77,95%) berukuran `300×300` piksel, sedangkan sisanya bervariasi (`295×221`, `499×332`, dll.), sehingga perlu di-resize ke ukuran standar `128×128` piksel.
4. **Duplikat & Potensi Leakage:** Ditemukan **1.138 citra duplikat identik (19,18%)** berdasarkan *MD5 hash*, serta struktur augmentasi rotasi dari **858 spesimen daun fisik**. Seluruh duplikat MD5 dihapus sebelum pemodelan, dan pembagian train/test dilakukan per grup spesimen daun (`leaf_group`).